<a id="encrypted-math-tutorial-bits"></a>
# Encrypted Math Tutorial: Bits

This tutorial covers `src/concrete_fhe_toolkit/math/bits.py`. While FHE naturally operates on integer data, sometimes you need true bitwise logic (e.g., bitwise AND, shift operations, or bit-level adders). This module provides these using tiny, heavily optimized lookup tables (LUTs) that execute extremely fast in FHE.

<a id="bitwise-logic-and-selectors"></a>
## Bitwise Logic and Selectors

In [ ]:
from concrete import fhe
from concrete_fhe_toolkit.math.bits import bit_not, bit_and, bit_or, bit_xor, bit_select

def test_logic(a: int, b: int):
    # These functions expect 0 or 1 as inputs
    return bit_not(a), bit_and(a, b), bit_or(a, b), bit_xor(a, b), bit_select(control=a, when_one=99, when_zero=88)

assert test_logic(1, 0) == (0, 0, 1, 1, 99)
assert test_logic(0, 1) == (1, 0, 1, 1, 88)
print("Cleartext logic passed!")

compiler = fhe.Compiler(test_logic, {"a": "encrypted", "b": "encrypted"})
inputset = [(0, 0), (1, 0), (0, 1), (1, 1)]
circuit = compiler.compile(inputset)

enc_res = circuit.encrypt_run_decrypt(1, 0)
assert enc_res == (0, 0, 1, 1, 99)
print("✅ Encrypted logic passed!")

<a id="integer-bit-conversions"></a>
## Integer <-> Bit Conversions

In [ ]:
from concrete_fhe_toolkit.math.bits import integer_to_bits, bits_to_unsigned

# 6 in 3-bit little-endian is (0, 1, 1)
bits = integer_to_bits(6, width=3)
assert bits == (0, 1, 1)

val = bits_to_unsigned(bits)
assert val == 6
print("✅ Cleartext Bit conversions passed!")
# Note: integer_to_bits usually operates on clear values to prepare bit-lists for FHE processing.

<a id="bit-shifting-and-rotation-shift_left_bits"></a>
## Bit Shifting and Rotation (`shift_left_bits`)

In [ ]:
from concrete_fhe_toolkit.math.bits import shift_left_bits, shift_right_bits

# [1, 1, 0] is 3. Shift left by 1 gives [0, 1, 1] which is 6.
shifted_left = shift_left_bits([1, 1, 0], amount=2)
assert shifted_left == (0, 0, 1)

shifted_right = shift_right_bits([0, 1, 1], amount=1)
assert shifted_right == (1, 1, 0)

print("✅ Bit shifts passed!")